# Paper extras: threshold ablation, cost, qualitative figure

These are the last results needed for the CVPR paper. Everything reuses what is already on Drive; there are no CSCI reruns.

| Cell | What | GPU? | Time |
|---|---|---|---|
| 1 | Load the saved pose checks and OSNet frame features (full MEVID run) | no | 1 min |
| 2 | **Threshold ablation:** all frames / person found / ≥25% / **≥50% (ours)** / ≥75% of the body visible. OSNet, on full MEVID and on MEVID-SC, 50 fair tests. Includes a check that the 50% rule reproduces the main run exactly | no | about 10 min |
| 3 | **Cost:** speed of the pose check vs OSNet and FastReID, on 4,096 real frames; total cost on full MEVID | yes | about 10 min |
| 4 | **Qualitative figure:** clips where CSCI was wrong with all frames and right with ours. Removed frames are outlined in red. Saved as PNG and PDF | no | about 10 min |

Results are saved to `MEVID-results/Paper-Extras`. Send the zip from cell 4, or paste the cell 2 and cell 3 tables.

In [ ]:
# 1. Setup (no GPU needed for cells 1-2). Loads the saved pose checks and
#    OSNet frame features of the full MEVID run, and the MEVID-SC mapping.
import os, glob, json, time, subprocess, sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])
CSCI_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI/test_templates_csci.npz',
                                 f'{SD}/**/CSCI/test_templates_csci.npz']))
RES = os.path.dirname(CSCI_DIR); DATA = os.path.dirname(RES)
FM = f'{RES}/FullMEVID-OcclusionAware'; OA = f'{RES}/OcclusionAware-SameClothes'
SC = f'{DATA}/MEVID-SameClothes'
OUT = f'{RES}/Paper-Extras'; os.makedirs(OUT, exist_ok=True)
MEVID_DIR = f'{DATA}/MEVID'
LOCAL = '/content/mevid'; os.makedirs(LOCAL, exist_ok=True)
subprocess.run(['unzip', '-oq', f'{MEVID_DIR}/mevid-v1-annotation-data.zip', '-d',
                f'{LOCAL}/ann'], check=True)
ann = lambda n: sorted(glob.glob(f'{LOCAL}/ann/**/{n}', recursive=True))[0]
names = [l.strip() for l in open(ann('test_name.txt')) if l.strip()]
info = np.loadtxt(ann('track_test_info.txt')).astype(np.int64)
z0 = np.load(f'{CSCI_DIR}/test_templates_csci.npz')
pid, oid, cam, qidx, gidx = z0['pid'], z0['oid'], z0['cam'], z0['qidx'], z0['gidx']
# MEVID-SC clips inside the full test, and the MEVID-SC queries
sc_idx = np.loadtxt(find([f'{SC}/original_index_test.txt', f'{SC}/**/original_index_test.txt'])).astype(np.int64)
sc_q = np.loadtxt(f'{SC}/annotations/query_IDX.txt').astype(np.int64)
assert len(sc_idx) == 1368
# checked frames and pose results, rebuilt exactly as in the full-MEVID run
check = []
for s, e in info[:, :2]:
    c = list(range(s, e + 1, 4))
    if c[-1] != e:
        c.append(e)
    check += c
K = np.unique(np.array(check))
done = {}
sc_names = [l.strip() for l in open(f'{SC}/annotations/test_name.txt') if l.strip()]
for f in sorted(glob.glob(f'{OA}/personcheck_[0-9]*.npz')):
    p = np.load(f)
    for k, fo, vi in zip(p['k'], p['found'], p['vis']):
        done[sc_names[k]] = (int(fo), float(vi))
for f in sorted(glob.glob(f'{FM}/personcheck_full_*.npz')):
    p = np.load(f)
    for n, fo, vi in zip(p['name'], p['found'], p['vis']):
        done[str(n)] = (int(fo), float(vi))
FOUND = np.array([done[names[k]][0] for k in K], np.int8)
VIS = np.array([done[names[k]][1] for k in K], np.float32)
OSN = np.load(f'{FM}/osnet_checked_frames.npy').astype(np.float32)
assert len(OSN) == len(K)
RANGES = [(np.searchsorted(K, s), np.searchsorted(K, e, 'right')) for s, e in info[:, :2]]
print(f'{len(info)} clips, {len(K):,} checked frames, MEVID-SC subset {len(sc_idx)} clips')
print('All results of this notebook are saved to:', OUT)


In [ ]:
# 2. Threshold ablation (OSNet; no GPU): how strict should "person
#    visible" be? Rules: all frames | person found | >=25% | >=50% (ours,
#    frozen) | >=75% of body keypoints visible. Evaluated on the full MEVID
#    test and on the MEVID-SC subset, same 50 fair tests as everywhere.
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
unit = lambda x: x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
RULES = {'all frames': None, 'person found': 0.0, 'visible >= 25%': 0.25,
         'visible >= 50% (ours)': 0.5, 'visible >= 75%': 0.75}
FP, REMOVED = {}, {}
for rule, thr in RULES.items():
    X = np.zeros((len(info), OSN.shape[1]), np.float32); rm = 0
    for t, ((s, e, *_), (lo, hi)) in enumerate(zip(info, RANGES)):
        f = OSN[lo:hi]
        if thr is None:
            X[t] = f.mean(0); continue
        okc = (FOUND[lo:hi] == 1) & (VIS[lo:hi] >= thr) if thr else (FOUND[lo:hi] == 1)
        fr = np.arange(s, e + 1)
        near = np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
        if okc[near].sum() < 8 or not okc.any():
            X[t] = f.mean(0)
        else:
            X[t] = f[okc].mean(0); rm += (~okc[near]).sum()
    FP[rule] = unit(X); REMOVED[rule] = 100 * rm / len(names)
ref = np.load(f'{FM}/fingerprints_OSNet.npz')
assert np.allclose(FP['visible >= 50% (ours)'], unit(ref['ours (pose: person visible)']), atol=1e-4), \
    'does not reproduce the main run'
print('check passed: the 50% rule reproduces the main OSNet run exactly')

def accuracy(X, q, g):
    S = X[q] @ X[g].T; aps, r1 = [], []
    for i, qq in enumerate(q):
        sp = pid[g] == pid[qq]; sc = cam[g] == cam[qq]
        good = np.flatnonzero(sp & ~sc); junk = np.flatnonzero(sp & sc)
        if not good.size:
            continue
        order = np.argsort(-S[i]); order = order[~np.isin(order, junk)]
        r = np.flatnonzero(np.isin(order, good))
        aps.append(np.mean((np.arange(len(r)) + 1) / (r + 1))); r1.append(float(r[0] == 0))
    return 100 * np.mean(aps), 100 * np.mean(r1)

def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True); _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    return 100 * M[r, cc].sum() / len(p), (M > 0).sum(1).mean()

def pp(lab, p):
    _, pi = np.unique(p, return_inverse=True); _, ci = np.unique(lab, return_inverse=True)
    cell = np.bincount(pi * (ci.max() + 1) + ci).astype(float); nc = np.bincount(ci).astype(float)
    same = (nc * (nc - 1)).sum()
    return 100 * (cell * (cell - 1)).sum() / same if same else 100.0

CUTS = np.geomspace(0.001, 1.5, 120)
def consistent(X, clips):
    P, C = pid[clips], cam[clips]
    people = np.unique(P); rng = np.random.default_rng(2027); out = []
    for s in tqdm(range(25), desc='   25 splits x 2 directions', leave=False):
        perm = rng.permutation(people); a = np.isin(P, perm[:len(people) // 2])
        G = [clips[a], clips[~a]]
        Ls = []
        for g in G:
            D = np.clip(1 - X[g] @ X[g].T, 0, None); D = (D + D.T) / 2; np.fill_diagonal(D, 0)
            Ls.append(linkage(squareform(D, checks=False), 'average'))
        for c, t in [(0, 1), (1, 0)]:
            best = CUTS[0]
            for cut in CUTS:
                if pp(fcluster(Ls[c], cut, 'distance'), pid[G[c]]) >= 90:
                    best = cut
            out.append(score(fcluster(Ls[t], best, 'distance'), pid[G[t]], cam[G[t]]))
    return np.mean(out, 0)

ROWS = []
for rule, X in tqdm(FP.items(), desc='rules'):
    for setname, clips, q in [('full MEVID', np.arange(len(info)), qidx),
                              ('MEVID-SC', sc_idx, sc_idx[sc_q])]:
        g = np.setdiff1d(clips, q)
        mAP, r1 = accuracy(X, q, g)
        idf1, ids = consistent(X, clips)
        ROWS.append({'set': setname, 'rule': rule, 'frames removed % (full)': round(REMOVED[rule], 1),
                     'IDF1 %': round(idf1, 2), 'IDs per person': round(ids, 2),
                     'mAP %': round(mAP, 2), 'Rank-1 %': round(r1, 2)})
ABL = pd.DataFrame(ROWS)
ABL.to_csv(f'{OUT}/ablation_threshold_osnet.csv', index=False)
display(ABL.set_index(['set', 'rule']))


In [ ]:
# 3. Cost (needs a GPU): speed of the pose check vs the ReID models, on
#    4,096 real MEVID frames read straight from the archive (no full unzip).
#    Model time only (images already in memory), fp16, typical batch sizes.
import tarfile, cv2, torch
from PIL import Image
from tqdm.auto import tqdm
frames = []
bar = tqdm(total=4096, desc='reading frames from the archive')
with tarfile.open(f'{MEVID_DIR}/mevid-v1-bbox-test.tgz', 'r|gz') as t:
    for m in t:
        if m.isfile() and m.name.endswith('.jpg'):
            frames.append(cv2.imdecode(np.frombuffer(t.extractfile(m).read(), np.uint8), 1))
            bar.update(1)
            if len(frames) >= 4096:
                break
bar.close()
print(len(frames), 'frames; GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'torchreid',
                'gdown', 'yacs', 'termcolor', 'tabulate'], check=True)

def timeit(fn, n_items, reps=2):
    fn(); torch.cuda.synchronize()                       # warm-up
    t0 = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return 1000 * (time.time() - t0) / (reps * n_items)  # ms per frame

COST = {}
from ultralytics import YOLO
from ultralytics.utils import LOGGER
LOGGER.setLevel('ERROR')            # hide repeated deprecation notes
pose = YOLO('yolo11m-pose.pt')
def run_pose():
    for i in range(0, len(frames), 128):
        pose.predict(frames[i:i + 128], imgsz=320, conf=0.25, verbose=False, half=True)
print('timing the pose check ...')
COST['pose check (YOLO11m-pose, 320 px)'] = timeit(run_pose, len(frames))

import torchvision.transforms as TV, torchreid, gdown
rgb = [Image.fromarray(f[:, :, ::-1]) for f in frames]
def batches(tf, bs):
    x = torch.stack([tf(im) for im in rgb])
    return [x[i:i + bs].cuda() for i in range(0, len(x), bs)]
m = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000, loss='softmax', pretrained=True)
ck = os.path.expanduser('~/.cache/torch/checkpoints/osnet_x1_0_msmt17.pth')
os.makedirs(os.path.dirname(ck), exist_ok=True)
for url in ['https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M',
            'https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x']:
    if not os.path.exists(ck):
        try: gdown.download(url, ck, quiet=True, fuzzy=True)
        except Exception as ex: print(ex)
torchreid.utils.load_pretrained_weights(m, ck); m = m.cuda().eval()
B = batches(TV.Compose([TV.Resize((256, 128)), TV.ToTensor(),
                        TV.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]), 512)
def run_os():
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x in B: m(x)
print('timing OSNet ...')
COST['OSNet x1_0'] = timeit(run_os, len(frames)); del m, B; torch.cuda.empty_cache()

FRP = '/content/fast-reid'
if not os.path.isdir(FRP):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/JDAI-CV/fast-reid.git', FRP], check=True)
    for f in [f'{FRP}/fastreid/data/build.py', f'{FRP}/fastreid/evaluation/testing.py']:
        s = open(f).read()
        s = s.replace('from collections import Mapping, OrderedDict',
                      'from collections import OrderedDict\nfrom collections.abc import Mapping')
        open(f, 'w').write(s.replace('from collections import Mapping', 'from collections.abc import Mapping'))
WFR = '/content/msmt_sbs_R101-ibn.pth'
if not os.path.exists(WFR):
    subprocess.run(['wget', '-q', '-O', WFR, 'https://github.com/JDAI-CV/fast-reid/releases/'
                    'download/v0.1.1/msmt_sbs_R101-ibn.pth'], check=True)
sys.path.insert(0, FRP)
from fastreid.config import get_cfg
from fastreid.modeling.meta_arch import build_model
from fastreid.utils.checkpoint import Checkpointer
cfg = get_cfg(); cfg.merge_from_file(f'{FRP}/configs/MSMT17/sbs_R101-ibn.yml')
cfg.MODEL.BACKBONE.PRETRAIN = False; cfg.freeze()
m = build_model(cfg).cuda().eval(); Checkpointer(m).load(WFR)
B = batches(TV.Compose([TV.Resize((384, 128)), TV.PILToTensor(), TV.Lambda(lambda t: t.float())]), 256)
def run_fr():
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x in B: m(x)
print('timing FastReID ...')
COST['FastReID SBS R101-IBN'] = timeit(run_fr, len(frames)); del m, B; torch.cuda.empty_cache()

n_frames, n_checked = len(names), len(K)
removed = int(np.load(f'{FM}/keep_summary.npz')['remove_all'].sum())
rows = [{'step': k, 'ms per frame': round(v, 3), 'frames per second': round(1000 / v)}
        for k, v in COST.items()]
C = pd.DataFrame(rows)
pose_min = COST['pose check (YOLO11m-pose, 320 px)'] * n_checked / 60000
print(f'GPU: {torch.cuda.get_device_name(0)}')
display(C)
print(f'\nFull MEVID test: {n_frames:,} frames; the pose check runs on every 4th '
      f'frame ({n_checked:,}) -> about {pose_min:.1f} min in total.')
print(f'Ours removes {removed:,} frames ({100 * removed / n_frames:.1f}%), so CSCI processes '
      f'{100 * removed / n_frames:.1f}% fewer frames. Measured on the L4 earlier: CSCI on our '
      f'filtered clips (982,256 frames) took 98 min, i.e. about {982256 / (98 * 60):.0f} frames/s; '
      f'all {n_frames:,} frames would take about {n_frames / (982256 / 98):.0f} min.')
C.to_csv(f'{OUT}/cost.csv', index=False)
json.dump({'gpu': torch.cuda.get_device_name(0), 'ms_per_frame': COST,
           'n_frames': n_frames, 'n_checked': n_checked, 'removed': removed,
           'pose_total_min': pose_min}, open(f'{OUT}/cost.json', 'w'), indent=2)


In [ ]:
# 4. Qualitative figure: clips where CSCI's first match was WRONG with all
#    frames and RIGHT with ours. Shows evenly spaced frames of each clip;
#    a red frame = removed by our pose check (person not visible).
#    Reads only the needed images from the archive (one pass, ~10 min).
import tarfile, cv2
import matplotlib.pyplot as plt, matplotlib.patches as mp
unit = lambda x: x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
XA = unit(z0['templates'])
XO = unit(np.load(f'{FM}/test_templates_csci_visible.npz')['templates'])
def top1(X, q):
    g = gidx[~((pid[gidx] == pid[q]) & (cam[gidx] == cam[q]))]
    return g[np.argmax(X[g] @ X[q])]
fixed = [q for q in qidx if pid[top1(XA, q)] != pid[q] and pid[top1(XO, q)] == pid[q]]
rm = np.load(f'{FM}/keep_summary.npz')['remove_all']
fixed = sorted(fixed, key=lambda q: -rm[q])[:4]
print(f'{sum(1 for q in qidx if pid[top1(XA, q)] != pid[q] and pid[top1(XO, q)] == pid[q])} '
      f'queries fixed by ours; showing {len(fixed)}')
SHOW, NEED = {}, set()
for q in fixed:
    s, e = info[q, :2]; lo, hi = RANGES[q]
    fr = np.linspace(s, e, 10).round().astype(int)
    near = lo + np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
    keep = (FOUND[near] == 1) & (VIS[near] >= 0.5)
    SHOW[q] = list(zip(fr, keep)); NEED |= {names[f] for f in fr}
from tqdm.auto import tqdm
IM = {}
found_bar = tqdm(total=len(NEED), desc='figure frames found')
scan_bar = tqdm(total=len(names), desc='archive scanned (images)', miniters=5000)
with tarfile.open(f'{MEVID_DIR}/mevid-v1-bbox-test.tgz', 'r|gz') as t:
    for m in t:
        if not m.isfile():
            continue
        scan_bar.update(1)
        b = os.path.basename(m.name)
        if b in NEED:
            IM[b] = cv2.imdecode(np.frombuffer(t.extractfile(m).read(), np.uint8), 1)[:, :, ::-1]
            found_bar.update(1)
            if len(IM) == len(NEED):
                break
scan_bar.close(); found_bar.close()
fig, ax = plt.subplots(len(fixed), 10, figsize=(16, 3.3 * len(fixed)))
ax = np.atleast_2d(ax)
for r, q in enumerate(fixed):
    for c, (f, k) in enumerate(SHOW[q]):
        a = ax[r, c]; a.axis('off')
        im = IM.get(names[f])
        if im is not None:
            a.imshow(im)
            if not k:
                a.add_patch(mp.Rectangle((0, 0), im.shape[1] - 1, im.shape[0] - 1, fill=False,
                                         edgecolor='#C0392B', linewidth=4))
        if c == 0:
            a.set_title(f'clip {q}: {rm[q]} of {info[q,1]-info[q,0]+1} frames removed',
                        loc='left', fontsize=10)
fig.tight_layout()
fig.savefig(f'{OUT}/qualitative_removed_frames.png', dpi=150)
fig.savefig(f'{OUT}/qualitative_removed_frames.pdf')
pd.DataFrame({'clip': fixed, 'person': pid[fixed], 'frames removed': rm[fixed],
              'clip length': info[fixed, 1] - info[fixed, 0] + 1}).to_csv(
    f'{OUT}/qualitative_clips.csv', index=False)
plt.show()
import zipfile
with zipfile.ZipFile('/content/Paper_Extras.zip', 'w') as zf:
    for f in glob.glob(f'{OUT}/*'):
        zf.write(f, os.path.basename(f))
print('Saved to', OUT, '\nZip: /content/Paper_Extras.zip')
